# GPU で画像生成（SDXL Turbo）

[SDXL Turbo](https://huggingface.co/stabilityai/sdxl-turbo) を Colab の GPU で使い、512×512 の画像を生成します。初回は数 GB のモデルをダウンロードします。モデルの利用条件は [モデルカードとライセンス](https://huggingface.co/stabilityai/sdxl-turbo) を確認してください。

**ランタイム → ランタイムのタイプを変更 → GPU** を選び、上から順に実行してください。別の GPU モデルを同じセッションで読み込んでいると VRAM が不足することがあります。

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU が見つかりません。Colab の「ランタイム → ランタイムのタイプを変更」で GPU を選んでから再実行してください。')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
print('空きVRAM: %.1f GiB' % (torch.cuda.mem_get_info()[0] / 1024**3))


## ライブラリを準備

In [ ]:
%pip -q install "diffusers>=0.30,<1" "transformers>=4.50,<5" accelerate safetensors


## モデルを読み込む

In [ ]:
from diffusers import AutoPipelineForText2Image

MODEL_ID = 'stabilityai/sdxl-turbo'
pipe = AutoPipelineForText2Image.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, variant='fp16'
).to('cuda')
pipe.set_progress_bar_config(disable=False)
print('読み込み完了:', MODEL_ID)


## 生成する

`PROMPT` を編集してください。英語の短い説明が扱いやすいモデルです。`STEPS` は 1〜4 に設定し、同じ画像を再現したい場合は `SEED` を固定します。SDXL Turbo では guidance scale を 0 にします。

In [ ]:
PROMPT = 'a quiet Japanese railway station at dusk, cinematic photography'
SEED = 42
STEPS = 2  # 1〜4

if not PROMPT.strip() or not 1 <= STEPS <= 4:
    raise ValueError('PROMPT は空にせず、STEPS は 1〜4 にしてください。')
generator = torch.Generator(device='cuda').manual_seed(SEED)
with torch.inference_mode():
    image = pipe(prompt=PROMPT, num_inference_steps=STEPS,
                 guidance_scale=0.0, width=512, height=512,
                 generator=generator).images[0]
display(image)


## PNG を保存（任意）

生成セルを再実行するたびに別ファイル名を使います。

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from google.colab import files
from uuid import uuid4

output_path = Path('/content') / f'sdxl_turbo_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}_{uuid4().hex[:6]}.png'
image.save(output_path)
print('保存先:', output_path)
files.download(str(output_path))
